# 2. Open RadDB object and filter

Tutorial 1 wrote an archive. This one reads it back and filters it down.

**`RadDB` is one class with two roles.**

| role | what it is |
|---|---|
| *archive-bound* | knows where an archive lives, and reads from it |
| *data-carrying*  | holds the gates you loaded, and narrows them down |

`open()` turns the first into the second. Every operation on a data-carrying
RadDB returns a **new** one, so calls chain and nothing is changed in place.

---

In [ ]:
from pathlib import Path

# --------------------------------------------------------------------------
# CONFIGURATION — edit these three paths to point at your own data
# --------------------------------------------------------------------------
# ARCHIVE_DIR must be the same archive tutorial 1 wrote.

MCH_DIR     = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/MCH_datatree_zarr").expanduser()
NEXRAD_DIR  = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree_zarr").expanduser()
ARCHIVE_DIR = Path("~/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive").expanduser()

print("MCH DataTrees   :", MCH_DIR)
print("NEXRAD DataTrees:", NEXRAD_DIR)
print("Archive         :", ARCHIVE_DIR)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import polars as pl
import raddb

In [ ]:
# This notebook stands on its own: build the archive if tutorial 1 has not run.
if not (ARCHIVE_DIR / "L" / "LUT").exists():
    print("building the archive (see tutorial 1) ...")
    raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=2056).archive(datatree_dir=MCH_DIR)
else:
    print("archive already present:", ARCHIVE_DIR)

## 1. `open()`: reading the archive

Reading never needs a CRS: it is recovered from the archive itself.

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)
rdf = db.open(radars="L")
rdf.head()

`open()` narrows *before* anything is loaded — the time range, the radars and the
columns are all pushed down into the Parquet scan, so you never pay for data you
did not ask for.

In [ ]:
# Only two moments, only radar L
small_df = db.open(radars="L", columns=["DBZH", "ZDR"])
print(f"small_df:\tcolumns: {small_df.columns()}")

# time period
day_df = db.open(radars="L", time_period=("2024-06-12", "2024-06-13"))
print(f"day_df:\t\tgates: {len(day_df):,}")

In [ ]:
# Filters can be pushed down at open() too, so filtered-out rows are never materialised
filtered_df = db.open(radars="L", filters={"var": "DBZH", "logic": ">", "threshold": 30})
print(f"before:\t{len(rdf):,} gates\t(with DBZH > 0 dBz)\nafter:\t{len(filtered_df):,}  gates\t(with DBZH > 30 dBz)")

## 2. What you are holding

The data lives in `.data` as a **polars** DataFrame. Polars is the backend
throughout RadDB (the read path, the LUT, the archive writer).

In [ ]:
print("type:", type(rdf.data).__name__)
print("shape:", rdf.data.shape)
rdf.data.head()

In [ ]:
print("radars    :", rdf.radars())
print("variables :", rdf.columns())
print("time range:", rdf.start_time(), "->", rdf.end_time())
print("lon/lat    :", [round(v, 3) for v in rdf.geographic_extent()])
print("archive CRS:", rdf.crs())          # recovered from the archive itself

## 3. `filter()`: threshold on values

A filter is a plain dict: `{"var", "logic", "threshold"}`

In [ ]:
rain = rdf.filter({"var": "DBZH", "logic": ">", "threshold": 20})
print(f"DBZH > 20: {len(rain):,} gates")

filt_df = rdf.filter([
    {"var": "DBZH",  "logic": ">",  "threshold": 20},
    {"var": "RHOHV", "logic": ">=", "threshold": 0.98},
    {"var": "ZDR",   "logic": ">",  "threshold": 4},
])
print(f"DBZH > 20, RHOHV >= 0.98, ZDR > 4 : {len(filt_df):,} gates")

## 4. `sel()`: select by label, xarray-style

Where `filter()` thresholds *values*, `sel()` selects by **coordinate**: a time, a
sweep, a range window, a longitude/latitude box. Scalars match exactly, `slice`
gives a closed interval, and a list matches any of its members.

In [ ]:
print("one sweep      :", f"{len(rdf.sel(sweep=1)):,}")
print("sweeps 1,2,3   :", f"{len(rdf.sel(sweep=[1, 2, 3])):,}")
print("range 10-50 km :", f"{len(rdf.sel(range=slice(10_000, 50_000))):,}")
print("a lon/lat box  :", f"{len(rdf.sel(lon=slice(8.6, 9.0), lat=slice(46.0, 46.4))):,}")

The clever part: `range`, `azimuth`, `elevation_angle`, `latitude`, `longitude`
and `altitude` are **not stored in the Parquet files** — they live once in the LUT.
`sel()` borrows the column it needs, evaluates the selection, and drops it again,
so selecting on geometry costs no storage.

In [ ]:
print("stored per gate:", rdf.columns())
print("also selectable :", ["range", "azimuth", "elevation_angle",
                            "latitude", "longitude", "altitude", "sweep"])

narrow = rdf.sel(sweep=1, range=slice(20_000, 60_000))
print(f"\nsweep 1, 20-60 km: {len(narrow):,} gates "
      f"(columns unchanged: {narrow.columns() == rdf.columns()})")

## 5. `add_feature()`: compute columns

`add_feature()` adds a column derived from the ones you already have and returns a
new RadDB, so it drops straight into a pipeline. The function receives the polars
frame; return a Series, a numpy array, or a polars expression.

In [ ]:
derived = (
    rdf.add_feature("DBZH_lin", lambda df: 10 ** (df["DBZH"] / 10))
       .add_feature("DBZH_dev", lambda df: df["DBZH"] - df["DBZH"].mean())
)
derived.head()

If you would rather work in plain polars or pandas, nothing stops you — `.data`
is an ordinary polars frame, and `to_pandas()` gives an ordinary pandas one.

In [ ]:
rdf.data.with_columns((pl.col("DBZH") - pl.col("ZDR")).alias("DIFF"))

df = rdf.to_pandas()
df["DIFF"] = df["DBZH"] - df["ZDR"]

print(f"rdf type: {type(rdf.data)}")
print(f"df  type: {type(df)}")
df.head()

## 6. Framework converter

Three converters, for three different framework.

In [ ]:
# pandas: with_geometry merges the per-gate coordinates from the LUT
df = rain.to_pandas(with_geometry=True)
print("to_pandas:", type(df))
print("columns:", list(df.columns))

In [ ]:
# with_polar_coords adds range / azimuth / elevation_angle as well.
# Off by default because they duplicate what the Cartesian columns already say.
print(list(rain.to_pandas(with_polar_coords=True).columns))

In [ ]:
# geopandas: point geometry per gate, ready for spatial joins or QGIS
gdf = rain.to_geopandas()
print("to_geopandas: ", type(gdf))
print("CRS:", gdf.crs)
gdf[["gate_id", "DBZH", "geometry"]].head()

In [ ]:
# DataTree: the full polar structure, for xarray workflows.
# A DataTree describes ONE volume: each sweep is an (azimuth x range) grid and
# time is a per-ray coordinate, so there is no dimension to stack volumes along.
# Choose which volume to rebuild; to_datatree() then NaN-fills the gates that
# were filtered out, restoring the complete azimuth x range grid.
volumes = rdf.data["volume_time"].unique().sort().to_list()
print(f"{len(volumes)} volumes loaded -> rebuilding the first one\n")

dt = rdf.to_datatree(timestep=volumes[0])
dt

---
**Next:** [3 — Areas of interest](03_area_of_interest.ipynb)